In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")


In [ ]:
import os

DRIVE_BASE = '/content/drive/MyDrive/Libri2Mix_Official_Test/sep_clean'

DRIVE_S1_CLEAN = os.path.join(DRIVE_BASE, 's1')
DRIVE_S1_NOISY_20 = os.path.join(DRIVE_BASE, 's1_noisy_20')
DRIVE_S1_NOISY_40 = os.path.join(DRIVE_BASE, 's1_noisy_40')

LOCAL_OUTPUT = '/content/libri1mix_output'
os.makedirs(LOCAL_OUTPUT, exist_ok=True)


for label, path in [('s1 (clean GT)', DRIVE_S1_CLEAN),
                     ('s1_noisy_20', DRIVE_S1_NOISY_20),
                     ('s1_noisy_40', DRIVE_S1_NOISY_40)]:
    if os.path.exists(path):
        n = len([f for f in os.listdir(path) if f.endswith('.wav')])
        print(f"  {label}: {n} wav files")
    else:
        print(f"  {label}: MISSING at {path}")


clean_ids = {f.replace('.wav', '') for f in os.listdir(DRIVE_S1_CLEAN) if f.endswith('.wav')}
noisy20_ids = {f.replace('.wav', '') for f in os.listdir(DRIVE_S1_NOISY_20) if f.endswith('.wav')}
noisy40_ids = {f.replace('.wav', '') for f in os.listdir(DRIVE_S1_NOISY_40) if f.endswith('.wav')}

SELECTED_IDS = sorted(clean_ids & noisy20_ids & noisy40_ids)
print(f"\nCommon IDs across all three folders: {len(SELECTED_IDS)}")


In [ ]:
# !pip install --force-reinstall torch torchaudio --index-url https://download.pytorch.org/whl/cu124
print("Colab already has PyTorch 2.6+cu124. Skip to Cell 4b.")


In [ ]:
!pip install -q huggingface_hub safetensors librosa soundfile ninja cmake

!pip uninstall -y causal-conv1d mamba-ssm 2>/dev/null

!CAUSAL_CONV1D_FORCE_BUILD=TRUE pip install causal-conv1d --no-build-isolation
!MAMBA_FORCE_BUILD=TRUE pip install mamba-ssm --no-build-isolation

print("\n=== Build complete. Now: Runtime -> Restart session, then run Cell 4c ===")


In [ ]:
import torch
print(f"PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

import causal_conv1d
print(f"causal-conv1d: {causal_conv1d.__version__}")

import mamba_ssm
print(f"mamba-ssm: {mamba_ssm.__version__}")
print("SUCCESS!")


In [ ]:
from huggingface_hub import snapshot_download
snapshot_download(repo_id="nvidia/RE-USE", local_dir="./REUSE", local_dir_use_symlinks=False)

import os
print("\nRE-USE files:")
for f in sorted(os.listdir('./REUSE')):
    print(f"  {f}")


In [ ]:
import os, sys, math
import torch
import torch.nn as nn
import torchaudio
import librosa
import numpy as np
from tqdm import tqdm

sys.path.insert(0, './REUSE')
from models.stfts import mag_phase_stft, mag_phase_istft
from models.generator_SEMamba_time_d4 import SEMamba
from utils.util import load_config, pad_or_trim_to_match
from huggingface_hub import hf_hub_download

RELU = nn.ReLU()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

config_path = hf_hub_download(repo_id='nvidia/RE-USE', filename='config.json')
cfg = load_config(config_path)
n_fft = cfg['stft_cfg']['n_fft']
hop_size = cfg['stft_cfg']['hop_size']
win_size = cfg['stft_cfg']['win_size']
compress_factor = cfg['model_cfg']['compress_factor']
sampling_rate = cfg['stft_cfg']['sampling_rate']

print("Loading RE-USE model...")
SE_model = SEMamba.from_pretrained('nvidia/RE-USE', cfg=cfg).to(device)
SE_model.eval()
print("Model loaded.")

def make_even(value):
    value = int(round(value))
    return value if value % 2 == 0 else value + 1

def enhance_short(wav_tensor, sr, model, device):
    wav = wav_tensor.to(device)
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    n_fft_s = make_even(n_fft * sr // sampling_rate)
    hop_s = make_even(hop_size * sr // sampling_rate)
    win_s = make_even(win_size * sr // sampling_rate)
    with torch.no_grad():
        noisy_mag, noisy_pha, _ = mag_phase_stft(
            wav, n_fft=n_fft_s, hop_size=hop_s, win_size=win_s,
            compress_factor=compress_factor, center=True, addeps=False
        )
        amp_g, pha_g, _ = model(noisy_mag, noisy_pha)
        mag = torch.expm1(RELU(amp_g))
        zero_portion = torch.sum(mag == 0, 1) / mag.shape[1]
        amp_g[:, :, (zero_portion > 0.5)[0]] = 0
        audio_g = mag_phase_istft(amp_g, pha_g, n_fft_s, hop_s, win_s, compress_factor)
        audio_g = pad_or_trim_to_match(wav.detach(), audio_g, pad_value=1e-8)
    return audio_g.cpu()

def enhance_chunked(wav_tensor, sr, model, device, chunk_sec=5.0, hop_portion=0.5):
    wav = wav_tensor.to(device)
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    chunk_size = int(chunk_sec * sr)
    hop_length = int(hop_portion * chunk_size)
    window = torch.hann_window(chunk_size).to(device)
    n_fft_s = make_even(n_fft * sr // sampling_rate)
    hop_s = make_even(hop_size * sr // sampling_rate)
    win_s = make_even(win_size * sr // sampling_rate)
    audio_enhanced = torch.zeros_like(wav)
    window_sum = torch.zeros_like(wav)
    n_chunks = max(1, math.ceil((wav.shape[1] - chunk_size) / hop_length) + 1)
    with torch.no_grad():
        for c in range(wav.shape[0]):
            noisy_wav = wav[c:c+1, :]
            for i in tqdm(range(n_chunks), desc=f"  ch{c}", leave=False):
                start = i * hop_length
                chunk = noisy_wav[:, start:start+chunk_size]
                noisy_mag, noisy_pha, _ = mag_phase_stft(
                    chunk, n_fft=n_fft_s, hop_size=hop_s, win_size=win_s,
                    compress_factor=compress_factor, center=True, addeps=False
                )
                amp_g, pha_g, _ = model(noisy_mag, noisy_pha)
                mag = torch.expm1(RELU(amp_g))
                zero_portion = torch.sum(mag == 0, 1) / mag.shape[1]
                amp_g[:, :, (zero_portion > 0.5)[0]] = 0
                audio_g = mag_phase_istft(amp_g, pha_g, n_fft_s, hop_s, win_s, compress_factor)
                audio_g = pad_or_trim_to_match(chunk.detach(), audio_g, pad_value=1e-8)
                seg_len = audio_g.shape[1]
                win_seg = window[:seg_len]
                audio_enhanced[c:c+1, start:start+seg_len] += audio_g * win_seg
                window_sum[c:c+1, start:start+seg_len] += win_seg
    nonzero = window_sum > 1e-8
    audio_enhanced[:, nonzero[0]] = audio_enhanced[:, nonzero[0]] / window_sum[:, nonzero[0]]
    return audio_enhanced.cpu()

LONG_THRESHOLD = 30

def enhance_auto(wav_tensor, sr, model, device):
    dur = wav_tensor.shape[-1] / sr
    if dur > LONG_THRESHOLD:
        return enhance_chunked(wav_tensor, sr, model, device)
    else:
        return enhance_short(wav_tensor, sr, model, device)

print("Enhancement functions ready.")


In [ ]:
import os

DRIVE_BASE = '/content/drive/MyDrive/Libri2Mix_Official_Test/sep_clean'
DRIVE_S1_CLEAN = os.path.join(DRIVE_BASE, 's1')
DRIVE_S1_NOISY_20 = os.path.join(DRIVE_BASE, 's1_noisy_20')
DRIVE_S1_NOISY_40 = os.path.join(DRIVE_BASE, 's1_noisy_40')

LOCAL_OUTPUT = '/content/libri1mix_output'
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

clean_ids = {f.replace('.wav', '') for f in os.listdir(DRIVE_S1_CLEAN) if f.endswith('.wav')}
noisy20_ids = {f.replace('.wav', '') for f in os.listdir(DRIVE_S1_NOISY_20) if f.endswith('.wav')}
noisy40_ids = {f.replace('.wav', '') for f in os.listdir(DRIVE_S1_NOISY_40) if f.endswith('.wav')}
SELECTED_IDS = sorted(clean_ids & noisy20_ids & noisy40_ids)

NOISE_CONDITIONS = {
    's1_noisy_20': ('20% WHAM noise', DRIVE_S1_NOISY_20),
    's1_noisy_40': ('40% WHAM noise', DRIVE_S1_NOISY_40),
}

print(f"Samples to process: {len(SELECTED_IDS)}")
print(f"Noise conditions: {list(NOISE_CONDITIONS.keys())}")


In [ ]:
import torchaudio, shutil
from tqdm import tqdm


for condition, (desc, noisy_dir) in NOISE_CONDITIONS.items():
    out_before = os.path.join(LOCAL_OUTPUT, condition, 'before')
    out_after = os.path.join(LOCAL_OUTPUT, condition, 'after_reuse')
    out_gt = os.path.join(LOCAL_OUTPUT, condition, 'clean_gt')

    for d in [out_before, out_after, out_gt]:
        os.makedirs(d, exist_ok=True)

    already_done = {f.replace('.wav', '') for f in os.listdir(out_after) if f.endswith('.wav')}
    to_process = [mid for mid in SELECTED_IDS if mid not in already_done]

    print(f"\n{'='*60}")
    print(f"  Enhancing: {condition} ({desc})")
    print(f"  Total: {len(SELECTED_IDS)}, already done: {len(already_done)}, remaining: {len(to_process)}")
    print(f"{'='*60}")

    skipped = 0
    for mix_id in tqdm(to_process, desc=condition):
        noisy_path = os.path.join(noisy_dir, f"{mix_id}.wav")
        gt_path = os.path.join(DRIVE_S1_CLEAN, f"{mix_id}.wav")

        if not os.path.exists(noisy_path):
            skipped += 1
            continue

        wav, sr = torchaudio.load(noisy_path)
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)

        enhanced = enhance_auto(wav, sr, SE_model, device)

        torchaudio.save(os.path.join(out_before, f"{mix_id}.wav"), wav, sr)
        torchaudio.save(os.path.join(out_after, f"{mix_id}.wav"), enhanced, sr)

        if os.path.exists(gt_path):
            shutil.copy2(gt_path, os.path.join(out_gt, f"{mix_id}.wav"))

    if skipped:
        print(f"  Skipped {skipped} missing files")


out_clean_before = os.path.join(LOCAL_OUTPUT, 'clean_s1', 'before')
out_clean_after = os.path.join(LOCAL_OUTPUT, 'clean_s1', 'after_reuse')
os.makedirs(out_clean_before, exist_ok=True)
os.makedirs(out_clean_after, exist_ok=True)

already_done_clean = {f.replace('.wav', '') for f in os.listdir(out_clean_after) if f.endswith('.wav')}
to_process_clean = [mid for mid in SELECTED_IDS if mid not in already_done_clean]

print(f"\n{'='*60}")
print(f"  Enhancing: clean_s1 (CLEAN single speaker through RE-USE)")
print(f"  Total: {len(SELECTED_IDS)}, already done: {len(already_done_clean)}, remaining: {len(to_process_clean)}")
print(f"{'='*60}")

skipped = 0
for mix_id in tqdm(to_process_clean, desc='clean_s1'):
    clean_path = os.path.join(DRIVE_S1_CLEAN, f"{mix_id}.wav")

    if not os.path.exists(clean_path):
        skipped += 1
        continue

    wav, sr = torchaudio.load(clean_path)
    if wav.shape[0] > 1:
        wav = wav.mean(dim=0, keepdim=True)

    enhanced = enhance_auto(wav, sr, SE_model, device)

    torchaudio.save(os.path.join(out_clean_before, f"{mix_id}.wav"), wav, sr)
    torchaudio.save(os.path.join(out_clean_after, f"{mix_id}.wav"), enhanced, sr)

if skipped:
    print(f"  Skipped {skipped} missing files")

print("\nAll enhancement done! (noisy_20 + noisy_40 + clean_s1)")


In [ ]:
import os, csv, torch, torchaudio, numpy as np

def si_sdr(estimate, reference):
    estimate = estimate - estimate.mean()
    reference = reference - reference.mean()
    dot = torch.sum(estimate * reference)
    s_target = dot * reference / (torch.sum(reference ** 2) + 1e-8)
    e_noise = estimate - s_target
    return (10 * torch.log10(
        torch.sum(s_target ** 2) / (torch.sum(e_noise ** 2) + 1e-8) + 1e-8
    )).item()

try:
    from pesq import pesq as pesq_fn
    HAS_PESQ = True
except ImportError:
    !pip install -q pesq
    try:
        from pesq import pesq as pesq_fn
        HAS_PESQ = True
    except:
        HAS_PESQ = False

try:
    from pystoi import stoi as stoi_fn
    HAS_STOI = True
except ImportError:
    !pip install -q pystoi
    try:
        from pystoi import stoi as stoi_fn
        HAS_STOI = True
    except:
        HAS_STOI = False

print(f"PESQ available: {HAS_PESQ}")
print(f"STOI available: {HAS_STOI}")

all_results = []

for condition in NOISE_CONDITIONS:
    before_dir = os.path.join(LOCAL_OUTPUT, condition, 'before')
    after_dir = os.path.join(LOCAL_OUTPUT, condition, 'after_reuse')
    gt_dir = os.path.join(LOCAL_OUTPUT, condition, 'clean_gt')

    if not os.path.exists(before_dir):
        print(f"Skipping {condition}: no output found")
        continue

    for fname in sorted(os.listdir(before_dir)):
        if not fname.endswith('.wav'):
            continue
        mix_id = fname.replace('.wav', '')

        gt_path = os.path.join(gt_dir, fname)
        if not os.path.exists(gt_path):
            continue

        before_wav, sr = torchaudio.load(os.path.join(before_dir, fname))
        after_wav, _ = torchaudio.load(os.path.join(after_dir, fname))
        gt_wav, _ = torchaudio.load(gt_path)

        before_wav = before_wav.squeeze(0)
        after_wav = after_wav.squeeze(0)
        gt_wav = gt_wav.squeeze(0)

        min_len = min(before_wav.shape[-1], after_wav.shape[-1], gt_wav.shape[-1])
        before_t = before_wav[:min_len]
        after_t = after_wav[:min_len]
        gt_t = gt_wav[:min_len]

        row = {
            'mixture_id': mix_id,
            'condition': condition,
            'sisdr_before': round(si_sdr(before_t, gt_t), 3),
            'sisdr_after_reuse': round(si_sdr(after_t, gt_t), 3),
        }

        if HAS_PESQ:
            try:
                row['pesq_before'] = round(pesq_fn(sr, gt_t.numpy(), before_t.numpy(), 'wb'), 3)
                row['pesq_after_reuse'] = round(pesq_fn(sr, gt_t.numpy(), after_t.numpy(), 'wb'), 3)
            except:
                row['pesq_before'] = None
                row['pesq_after_reuse'] = None

        if HAS_STOI:
            try:
                row['stoi_before'] = round(stoi_fn(gt_t.numpy(), before_t.numpy(), sr, extended=False), 4)
                row['stoi_after_reuse'] = round(stoi_fn(gt_t.numpy(), after_t.numpy(), sr, extended=False), 4)
            except:
                row['stoi_before'] = None
                row['stoi_after_reuse'] = None

        all_results.append(row)

csv_path = os.path.join(LOCAL_OUTPUT, 'libri1mix_reuse_metrics.csv')
if all_results:
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=all_results[0].keys())
        writer.writeheader()
        writer.writerows(all_results)
    print(f"Saved {len(all_results)} results to {csv_path}")


In [ ]:
import numpy as np

print(f"{'='*70}")
print(f"  RE-USE on Single-Speaker Noisy LibriMix (3000 samples)")
print(f"{'='*70}")

for condition, (desc, _) in NOISE_CONDITIONS.items():
    cond_results = [r for r in all_results if r['condition'] == condition]
    if not cond_results:
        continue

    sisdrs_before = [r['sisdr_before'] for r in cond_results]
    sisdrs_after = [r['sisdr_after_reuse'] for r in cond_results]

    print(f"\n  {condition} ({desc}) -- {len(cond_results)} samples")
    print(f"  {'-'*55}")
    print(f"    SI-SDR Before (noisy):     mean={np.mean(sisdrs_before):>7.2f}  med={np.median(sisdrs_before):>7.2f}")
    print(f"    SI-SDR After  (RE-USE):    mean={np.mean(sisdrs_after):>7.2f}  med={np.median(sisdrs_after):>7.2f}")
    print(f"    SI-SDR Improvement:        {np.mean(sisdrs_after) - np.mean(sisdrs_before):>+7.2f} dB")

    if HAS_PESQ:
        pesqs_b = [r['pesq_before'] for r in cond_results if r.get('pesq_before') is not None]
        pesqs_a = [r['pesq_after_reuse'] for r in cond_results if r.get('pesq_after_reuse') is not None]
        if pesqs_b:
            print(f"    PESQ Before:               mean={np.mean(pesqs_b):>7.3f}")
            print(f"    PESQ After:                mean={np.mean(pesqs_a):>7.3f}")
            print(f"    PESQ Improvement:          {np.mean(pesqs_a) - np.mean(pesqs_b):>+7.3f}")

    if HAS_STOI:
        stois_b = [r['stoi_before'] for r in cond_results if r.get('stoi_before') is not None]
        stois_a = [r['stoi_after_reuse'] for r in cond_results if r.get('stoi_after_reuse') is not None]
        if stois_b:
            print(f"    STOI Before:               mean={np.mean(stois_b):>7.4f}")
            print(f"    STOI After:                mean={np.mean(stois_a):>7.4f}")
            print(f"    STOI Improvement:          {np.mean(stois_a) - np.mean(stois_b):>+7.4f}")

print(f"\n{'='*70}")


In [ ]:
import os, csv, numpy as np

DRIVE_2SPK_CSV = '/content/drive/MyDrive/AD-FlowTSE_checkpoints/reuse_experiment_output/librimix_reuse_metrics.csv'

print(f"{'='*70}")
print(f"  COMPARISON: Single-Speaker (1spk+noise) vs Two-Speaker (2spk mix)")
print(f"{'='*70}")

for condition, (desc, _) in NOISE_CONDITIONS.items():
    cond_results = [r for r in all_results if r['condition'] == condition]
    if not cond_results:
        continue
    sisdrs_b = [r['sisdr_before'] for r in cond_results]
    sisdrs_a = [r['sisdr_after_reuse'] for r in cond_results]
    print(f"\n  [1spk] {condition}: SI-SDR {np.mean(sisdrs_b):.2f} -> {np.mean(sisdrs_a):.2f} ({np.mean(sisdrs_a)-np.mean(sisdrs_b):+.2f} dB)")
    if HAS_PESQ:
        pesqs_b = [r['pesq_before'] for r in cond_results if r.get('pesq_before') is not None]
        pesqs_a = [r['pesq_after_reuse'] for r in cond_results if r.get('pesq_after_reuse') is not None]
        if pesqs_b:
            print(f"           {condition}: PESQ  {np.mean(pesqs_b):.3f} -> {np.mean(pesqs_a):.3f} ({np.mean(pesqs_a)-np.mean(pesqs_b):+.3f})")

if os.path.exists(DRIVE_2SPK_CSV):
    with open(DRIVE_2SPK_CSV) as f:
        reader = csv.DictReader(f)
        two_spk = list(reader)

    sisdrs_b2 = [float(r['sisdr_before']) for r in two_spk]
    sisdrs_a2 = [float(r['sisdr_after_reuse']) for r in two_spk]
    print(f"\n  [2spk] mix_both:      SI-SDR {np.mean(sisdrs_b2):.2f} -> {np.mean(sisdrs_a2):.2f} ({np.mean(sisdrs_a2)-np.mean(sisdrs_b2):+.2f} dB)")

    pesqs_b2 = [float(r['pesq_before']) for r in two_spk if r.get('pesq_before') and r['pesq_before'] != 'None']
    pesqs_a2 = [float(r['pesq_after_reuse']) for r in two_spk if r.get('pesq_after_reuse') and r['pesq_after_reuse'] != 'None']
    if pesqs_b2:
        print(f"           mix_both:      PESQ  {np.mean(pesqs_b2):.3f} -> {np.mean(pesqs_a2):.3f} ({np.mean(pesqs_a2)-np.mean(pesqs_b2):+.3f})")

    print(f"\n  Key insight: RE-USE is a general speech enhancement model.")
    print(f"  It should work much better on 1spk+noise (its intended use case)")
    print(f"  than on 2spk mixtures (which need target speaker extraction).")
else:
    print(f"\n  2-speaker results not found at {DRIVE_2SPK_CSV}")
    print(f"  Upload the previous reuse_experiment_output/ to compare.")

print(f"\n{'='*70}")


In [ ]:
import os, shutil, numpy as np

N_SAVE = 10

for condition, (desc, _) in NOISE_CONDITIONS.items():
    cond_results = [r for r in all_results if r['condition'] == condition]
    if not cond_results:
        continue

    for r in cond_results:
        r['sisdr_improvement'] = r['sisdr_after_reuse'] - r['sisdr_before']

    sorted_by_improvement = sorted(cond_results, key=lambda r: r['sisdr_improvement'])
    worst = sorted_by_improvement[:N_SAVE]
    best = sorted_by_improvement[-N_SAVE:][::-1]

    for tag, samples in [('best_10', best), ('worst_10', worst)]:
        save_dir = os.path.join(LOCAL_OUTPUT, condition, tag)
        os.makedirs(save_dir, exist_ok=True)

        for rank, r in enumerate(samples, 1):
            mix_id = r['mixture_id']
            sample_dir = os.path.join(save_dir, f"rank{rank:02d}_{mix_id}")
            os.makedirs(sample_dir, exist_ok=True)

            before_src = os.path.join(LOCAL_OUTPUT, condition, 'before', f"{mix_id}.wav")
            after_src = os.path.join(LOCAL_OUTPUT, condition, 'after_reuse', f"{mix_id}.wav")
            gt_src = os.path.join(LOCAL_OUTPUT, condition, 'clean_gt', f"{mix_id}.wav")

            for src, name in [(before_src, '1_before_noisy.wav'),
                              (after_src, '2_after_reuse.wav'),
                              (gt_src, '3_ground_truth_clean.wav')]:
                if os.path.exists(src):
                    shutil.copy2(src, os.path.join(sample_dir, name))

            info_path = os.path.join(sample_dir, 'info.txt')
            with open(info_path, 'w') as f:
                f.write(f"Condition: {condition} ({desc})\n")
                f.write(f"Mixture ID: {mix_id}\n")
                f.write(f"SI-SDR before: {r['sisdr_before']:.3f} dB\n")
                f.write(f"SI-SDR after:  {r['sisdr_after_reuse']:.3f} dB\n")
                f.write(f"Improvement:   {r['sisdr_improvement']:+.3f} dB\n")
                if r.get('pesq_before') is not None:
                    f.write(f"PESQ before:   {r['pesq_before']:.3f}\n")
                    f.write(f"PESQ after:    {r['pesq_after_reuse']:.3f}\n")
                if r.get('stoi_before') is not None:
                    f.write(f"STOI before:   {r['stoi_before']:.4f}\n")
                    f.write(f"STOI after:    {r['stoi_after_reuse']:.4f}\n")

    print(f"\n{condition} ({desc}):")
    print(f"  Best 10 (most improved):")
    for rank, r in enumerate(best, 1):
        print(f"    {rank}. {r['mixture_id']}: {r['sisdr_improvement']:+.2f} dB ({r['sisdr_before']:.1f} -> {r['sisdr_after_reuse']:.1f})")
    print(f"  Worst 10 (most degraded):")
    for rank, r in enumerate(worst, 1):
        print(f"    {rank}. {r['mixture_id']}: {r['sisdr_improvement']:+.2f} dB ({r['sisdr_before']:.1f} -> {r['sisdr_after_reuse']:.1f})")

print(f"\nAudio saved to {LOCAL_OUTPUT}/<condition>/best_10/ and worst_10/")


In [ ]:
import shutil, os

DRIVE_OUT = '/content/drive/MyDrive/AD-FlowTSE_checkpoints/reuse_libri1mix_output'

if os.path.exists(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree(LOCAL_OUTPUT, DRIVE_OUT)

total = 0
for root, dirs, files in os.walk(DRIVE_OUT):
    total += len(files)

print(f"Saved {total} files to {DRIVE_OUT}")
print(f"\nStructure:")
for root, dirs, files in os.walk(DRIVE_OUT):
    level = root.replace(DRIVE_OUT, '').count(os.sep)
    indent = '  ' * level
    n_wav = len([f for f in files if f.endswith('.wav')])
    n_csv = len([f for f in files if f.endswith('.csv')])
    n_png = len([f for f in files if f.endswith('.png')])
    parts = []
    if n_wav: parts.append(f'{n_wav} wav')
    if n_csv: parts.append(f'{n_csv} csv')
    if n_png: parts.append(f'{n_png} png')
    suffix = f" ({', '.join(parts)})" if parts else ''
    print(f"{indent}{os.path.basename(root)}/{suffix}")

print(f"\nDownload this folder to: AD-FlowTSE/reuse_libri1mix_experiment/output/")
print(f"\nExpected structure for voice quality evaluation:")
print(f"  output/clean_s1/before/      <- clean single speaker (original)")
print(f"  output/clean_s1/after_reuse/  <- clean single speaker after RE-USE")
print(f"  output/s1_noisy_20/before/   <- noisy 20% (original)")
print(f"  output/s1_noisy_20/after_reuse/ <- noisy 20% after RE-USE")
print(f"  output/s1_noisy_40/before/   <- noisy 40% (original)")
print(f"  output/s1_noisy_40/after_reuse/ <- noisy 40% after RE-USE")
